# 01 · Nighttime light measures for every survey region

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/fundidayyan-afk/Measuring-Electrification-from-Space/blob/main/notebooks/01_light_measures.ipynb)

**What this notebook does.** For every DHS region (all eight surveys, every boundary level), each whole country, and every VIIRS year from 2013 on, it computes average brightness, the share of land that is lit, the share of **people** living in lit pixels (WorldPop), and a cloud-cover quality flag. First it sets the "lit" cutoff from large protected areas where nobody lives (rule fixed in `docs/analysis_plan.md`).

**How it runs.** The heavy work is sent to Earth Engine as background jobs that save CSV files to your Google Drive, in a folder named `electrification_from_space`. While they run, this notebook shows a status table that refreshes every 30 seconds. At the end, it combines the files and downloads `light_measures.csv` to your computer.

**How to use it: run one cell at a time** with the ▶ button on the left of each cell, top to bottom. Wait for each cell to finish (the ▶ stops spinning and a number appears in the brackets) before starting the next.

**Total time:** about 5 minutes of clicking, plus 20 to 60 minutes of waiting while Earth Engine works. You can leave the tab open and come back.

## Step 1. Sign in (two pop-ups)

When you run this cell:
1. A box asks to **allow this notebook to access your Google credentials**. Click **Allow**, choose **fundidayyan@gmail.com**, and allow again.
2. A second box asks to **connect to Google Drive**. Choose **fundidayyan@gmail.com** again and allow.

If nothing seems to happen, look for a blocked pop-up icon at the right end of your browser's address bar and allow pop-ups for colab.research.google.com.

In [ ]:
PROJECT = "electrification-from-space"
FOLDER = "electrification_from_space"        # Google Drive folder for the outputs

from google.colab import auth, drive
print("Waiting for you to approve the sign-in pop-up...")
auth.authenticate_user()
print("Signed in. Now approve the Google Drive pop-up...")
drive.mount("/content/drive")

import ee, google.auth
try:
    ee.Initialize(project=PROJECT)
except Exception:
    creds, _ = google.auth.default(scopes=["https://www.googleapis.com/auth/cloud-platform",
                                           "https://www.googleapis.com/auth/earthengine"])
    ee.Initialize(credentials=creds, project=PROJECT)

ee.data.setDeadline(120_000)   # quick requests give up after 2 minutes instead of hanging
try:
    ee.Number(1).getInfo()
    print("\nConnected to Earth Engine project:", PROJECT)
except Exception as e:
    msg = str(e)
    print("\nCould not connect. Google said:\n ", msg, "\n")
    low = msg.lower()
    if "regist" in low:
        print("FIX: register the project for Earth Engine (Unpaid usage > Academia & Research):")
        print(f"  https://code.earthengine.google.com/register?project={PROJECT}")
    elif "has not been used" in low or "disabled" in low:
        print("FIX: click ENABLE here, wait a minute, then run this cell again:")
        print(f"  https://console.cloud.google.com/apis/library/earthengine.googleapis.com?project={PROJECT}")
    elif "permission" in low or "403" in low:
        print("FIX: the account you chose does not own the project. Click Runtime > Disconnect and delete runtime,")
        print("  then run this cell again and pick fundidayyan@gmail.com in both pop-ups.")
    else:
        print("Copy this message into the chat with Claude.")
    raise

## Step 2. Load the region boundaries from GitHub (about 1 minute)

In [ ]:
!rm -rf repo && git clone -q --depth 1 https://github.com/fundidayyan-afk/Measuring-Electrification-from-Space.git repo
!pip -q install geopandas

import glob, os, time
import geopandas as gpd
import pandas as pd
from IPython.display import clear_output, display

TOL = 0.002    # simplify outlines by about 220 m (finer than a 460 m VIIRS pixel)
KEEP = ["SVYID", "CNTRYNAMEE", "REG_ID", "REGCODE", "REGNAME", "DHSREGEN", "LEVELRNK"]

layers = {}
for f in sorted(glob.glob("repo/data/boundaries/*.geojson")):
    name = os.path.basename(f).replace(".geojson", "")
    survey, layer = name.split("_", 1)
    g = gpd.read_file(f).to_crs(4326)
    g["geometry"] = g.geometry.simplify(TOL, preserve_topology=True)
    g["survey"], g["layer"] = survey, layer
    g["iso3"] = {"KE": "KEN", "TZ": "TZA", "UG": "UGA", "RW": "RWA"}[survey[:2]]
    g["uid"] = survey + "|" + layer + "|" + g["REG_ID"].astype(str)
    layers[name] = g[[k for k in KEEP if k in g] + ["survey", "layer", "iso3", "uid", "geometry"]]

# Whole-country outlines: merge the regions of each country's latest survey
NATIONAL_FROM = {"KEN": "KE2022DHS_sdr_subnational_boundaries", "TZA": "TZ2022DHS_sdr_subnational_boundaries2",
                 "UGA": "UG2024MIS_sdr_subnational_boundaries", "RWA": "RW2019DHS_sdr_subnational_boundaries"}
nat = []
for iso3, src in NATIONAL_FROM.items():
    d = layers[src].dissolve()[["geometry"]]
    d["geometry"] = d.geometry.simplify(TOL * 2, preserve_topology=True)
    d["survey"], d["layer"], d["iso3"], d["uid"] = "national", "national", iso3, "national|" + iso3
    nat.append(d)
layers["national"] = gpd.GeoDataFrame(pd.concat(nat, ignore_index=True), crs=4326)

def to_ee(gdf):
    feats = []
    for _, row in gdf.iterrows():
        props = {k: (None if pd.isna(v) else (v.item() if hasattr(v, "item") else v))
                 for k, v in row.drop("geometry").items()}
        feats.append(ee.Feature(ee.Geometry(row.geometry.__geo_interface__, "EPSG:4326", False), props))
    return ee.FeatureCollection(feats)

for k, g in layers.items():
    print(f"{k:45s} {len(g):3d} regions")

## Step 3. Set up the light and population data (a few seconds)

In [ ]:
VIIRS = ee.ImageCollection("NOAA/VIIRS/DNB/ANNUAL_V22")
POP = ee.ImageCollection("WorldPop/GP/100m/R2025A/v1/pop")
ISO3 = ["KEN", "TZA", "UGA", "RWA"]
POP_FIRST_YEAR = 2015          # WorldPop R2025A starts in 2015; 2013-2014 use the 2015 grid
SCALE = 463.83                 # VIIRS pixel size in metres
viirs_proj = VIIRS.first().select("average").projection()
pop_proj = POP.first().projection()

years = VIIRS.aggregate_array("system:time_start").map(lambda t: ee.Date(t).get("year")).getInfo()
YEARS = sorted({int(y) for y in years if int(y) >= 2013})   # 2012 is a partial-year composite
print("VIIRS years used:", YEARS)

def viirs(y):
    return ee.Image(VIIRS.filter(ee.Filter.calendarRange(y, y, "year")).first())

def pop_on_viirs_grid(y):
    """People per 100 m WorldPop cell, added up to the ~460 m VIIRS grid."""
    y = max(y, POP_FIRST_YEAR)
    img = (POP.filter(ee.Filter.rangeContains("year", y - 0.5, y + 0.5))   # year is stored as 2015.0
              .filter(ee.Filter.inList("country", ISO3))
              .mosaic().select("population").unmask(0)
              .setDefaultProjection(pop_proj))          # mosaics lose their pixel grid; restore it
    return (img.reduceResolution(ee.Reducer.sum(), maxPixels=1024)
               .reproject(viirs_proj).rename("pop"))

def wait_for(tasks, every=30):
    """Show a status table for Earth Engine jobs until all have finished."""
    while True:
        rows = []
        for t in tasks:
            s = t.status()
            rows.append({"job": s.get("description"), "state": s.get("state"),
                         "error": s.get("error_message", "")})
        table = pd.DataFrame(rows)
        clear_output(wait=True)
        print(time.strftime("%H:%M:%S"), "- refreshes every", every, "seconds")
        display(table)
        if table["state"].isin(["COMPLETED", "FAILED", "CANCELLED"]).all():
            failed = table[table["state"] != "COMPLETED"]
            if len(failed):
                raise RuntimeError("Some jobs failed (see the error column above). "
                                   "Copy the table into the chat with Claude.")
            else:
                print("All jobs finished.")
            return table
        time.sleep(every)

def read_from_drive(prefix, tries=40):
    """Drive can take a minute to show a new file; keep checking."""
    path = f"/content/drive/MyDrive/{FOLDER}/{prefix}.csv"
    for _ in range(tries):
        if os.path.exists(path):
            return pd.read_csv(path)
        time.sleep(15)
    raise FileNotFoundError(f"{path} did not appear. Check the {FOLDER} folder in Google Drive.")

print("Ready.")

## Step 4. Background threshold (one job, about 5 to 15 minutes)

Pixels in large, strictly protected areas (IUCN Ia, Ib, II; over 1,000 km²) with no people should be dark, so any light recorded there is noise. For each year we take the 99th percentile of that noise; the **highest** yearly value becomes the cutoff for "lit".

In [ ]:
# Quick checks first, so any problem shows up in seconds rather than after a long job
n_pop = POP.filter(ee.Filter.rangeContains("year", 2014.5, 2015.5)).filter(ee.Filter.inList("country", ISO3)).size().getInfo()
print("WorldPop 2015 images for the four countries:", n_pop, "(expect 4)")
assert n_pop > 0, "WorldPop filter matched nothing; send this message to Claude"
print("VIIRS bands:", viirs(YEARS[0]).bandNames().getInfo())

wdpa = (ee.FeatureCollection("WCMC/WDPA/current/polygons")
        .filter(ee.Filter.inList("ISO3", ISO3))
        .filter(ee.Filter.inList("IUCN_CAT", ["Ia", "Ib", "II"]))
        .filter(ee.Filter.inList("MARINE", ["0", 0]))
        .filter(ee.Filter.gt("GIS_AREA", 1000)))
n_pa = wdpa.size().getInfo()
print("Protected areas used:", n_pa)
print("  ", sorted(wdpa.aggregate_array("NAME").getInfo()))
assert n_pa > 0, "No protected areas matched the filters; send this message to Claude"

# Turn the protected areas into a 1/0 map instead of merging their outlines (much lighter)
pa_mask = ee.Image.constant(1).paint(wdpa, 0).eq(0)
STUDY_BOX = ee.Geometry.Rectangle([28.5, -12.0, 42.5, 5.5], "EPSG:4326", False)

def noise_stats(y):
    keep = pa_mask.And(pop_on_viirs_grid(y).lte(0))
    noise = viirs(y).select("average").updateMask(keep)
    d = noise.reduceRegion(
        reducer=ee.Reducer.percentile([50, 95, 99]).combine(ee.Reducer.count(), sharedInputs=True),
        geometry=STUDY_BOX, crs=viirs_proj, scale=SCALE, maxPixels=1e10, tileScale=4)
    return ee.Feature(None, d).set("year", y)

print("Checks passed. Starting the threshold job...")

thr_task = ee.batch.Export.table.toDrive(
    collection=ee.FeatureCollection([noise_stats(y) for y in YEARS]),
    description="light_threshold", folder=FOLDER, fileNamePrefix="light_threshold", fileFormat="CSV",
    selectors=["year", "average_p50", "average_p95", "average_p99", "average_count"])
thr_task.start()
wait_for([thr_task])

threshold_table = read_from_drive("light_threshold")
display(threshold_table)
THRESHOLD = float(threshold_table["average_p99"].max())
print(f"\nLit threshold = {THRESHOLD:.3f} nW/cm²/sr")

## Step 5. Light measures for every region and year (14 jobs, about 15 to 45 minutes)

One job per boundary file plus one for the whole countries. The table refreshes every 30 seconds. Jobs show READY (waiting in line), then RUNNING, then COMPLETED. You can close the laptop lid only if the notebook keeps running; safest is to leave the tab open.

In [ ]:
COLS = KEEP + ["survey", "layer", "iso3", "uid", "year",
               "rad", "lit", "lit_eog", "pop", "litpop", "litpop_eog", "cvg", "lowcvg"]

def year_measures(fc, y):
    v = viirs(y)
    rad = v.select("average_masked").unmask(0)          # EOG-masked radiance; masked pixels count as dark
    lit = v.select("average").gt(THRESHOLD)              # our threshold
    lit_eog = rad.gt(0)                                  # EOG's own mask (robustness)
    pop = pop_on_viirs_grid(y)
    cvg = v.select("cf_cvg")
    img = ee.Image.cat([rad.rename("rad"), lit.rename("lit"), lit_eog.rename("lit_eog"), pop.rename("pop"),
                        pop.multiply(lit).rename("litpop"), pop.multiply(lit_eog).rename("litpop_eog"),
                        cvg.rename("cvg"), cvg.lt(10).rename("lowcvg")]).toFloat()
    return (img.reduceRegions(collection=fc, reducer=ee.Reducer.mean(), crs=viirs_proj, scale=SCALE, tileScale=4)
               .map(lambda f: f.set("year", y)))

tasks = []
for name, g in layers.items():
    fc = to_ee(g)
    coll = ee.FeatureCollection([year_measures(fc, y) for y in YEARS]).flatten()
    t = ee.batch.Export.table.toDrive(collection=coll, description=f"light_{name}", folder=FOLDER,
                                      fileNamePrefix=f"light_{name}", fileFormat="CSV", selectors=COLS)
    t.start()
    tasks.append(t)
print(len(tasks), "jobs started")
status = wait_for(tasks)

## Step 6. Combine, check, and download (1 minute)

In [ ]:
import numpy as np

parts = [read_from_drive(f"light_{name}") for name in layers]
df = pd.concat(parts, ignore_index=True)

# Means over pixels -> shares. Lit population share = mean(people in lit pixels) / mean(people).
df["mean_rad"] = df["rad"]
df["log_rad"] = np.log1p(df["mean_rad"])
df["lit_land_share"] = df["lit"]
df["lit_land_share_eog"] = df["lit_eog"]
df["lit_pop_share"] = df["litpop"] / df["pop"].replace(0, np.nan)
df["lit_pop_share_eog"] = df["litpop_eog"] / df["pop"].replace(0, np.nan)
df["pop_per_pixel"] = df["pop"]
df["cf_cvg_mean"] = df["cvg"]
df["low_cvg_share"] = df["lowcvg"]
df["threshold"] = THRESHOLD
df["viirs_version"] = "NOAA/VIIRS/DNB/ANNUAL_V22"
df["pop_version"] = "WorldPop/GP/100m/R2025A/v1/pop"
df["run_date"] = pd.Timestamp.today().date().isoformat()

keep = ["uid", "survey", "layer", "iso3", "SVYID", "CNTRYNAMEE", "REG_ID", "REGCODE", "REGNAME", "DHSREGEN",
        "LEVELRNK", "year", "mean_rad", "log_rad", "lit_land_share", "lit_pop_share", "lit_land_share_eog",
        "lit_pop_share_eog", "pop_per_pixel", "cf_cvg_mean", "low_cvg_share", "threshold",
        "viirs_version", "pop_version", "run_date"]
df = df[[k for k in keep if k in df]].sort_values(["survey", "layer", "REGNAME", "year"], na_position="last")

for col in ["lit_land_share", "lit_pop_share", "lit_land_share_eog", "lit_pop_share_eog"]:
    bad = ((df[col] < 0) | (df[col] > 1)).sum()
    print(f"{col:20s} missing={df[col].isna().sum():4d}  out of range={bad}")
print(len(df), "rows\n")
print("Share of people living in lit pixels, whole countries:")
display(df[df.layer == "national"].pivot(index="year", columns="iso3", values="lit_pop_share").round(3))

df.to_csv("light_measures.csv", index=False)
threshold_table.to_csv("light_threshold.csv", index=False)
from google.colab import files
files.download("light_measures.csv")
files.download("light_threshold.csv")

**Last step:** send `light_measures.csv` and `light_threshold.csv` to Claude in the chat. (Copies also stay in your Google Drive folder `electrification_from_space`.)